In [7]:
import pandas as pd
import numpy as np

In [8]:
df = pd.read_csv("accepted_reefree_structures.csv")
df

,candidate_formula,prototype_id,substitution,w_letter,group_size,max_mismatch,status,generated_cif,n_neighbor_checks
0,Fe7Sn6,mp-672392,Dy->Fe,j,12,0.097871,accepted_radiusmatch_ok,outputs/wyckoff_from_candidates_radius_mismatc...,76
1,Co16Si2B5,mp-16085,Nd->Co,d,4,0.096480,accepted_radiusmatch_ok,outputs/wyckoff_from_candidates_radius_mismatc...,40
2,AlFe19,mp-1211934,La->Fe,l,24,0.137196,accepted_radiusmatch_ok,outputs/wyckoff_from_candidates_radius_mismatc...,144
3,Mn6AlFe13,mp-1211934,La->Mn,l,24,0.137196,accepted_radiusmatch_ok,outputs/wyckoff_from_candidates_radius_mismatc...,144
4,AlCr6Fe13,mp-1211934,La->Cr,l,24,0.137196,accepted_radiusmatch_ok,outputs/wyckoff_from_candidates_radius_mismatc...,144
...,...,...,...,...,...,...,...,...,...
89,Fe3Rh,mp-1189512,Eu->Fe,d,8,0.125733,accepted_radiusmatch_ok,outputs/wyckoff_from_candidates_radius_mismatc...,16
90,Co3Rh,mp-1189512,Eu->Co,d,8,0.146580,accepted_radiusmatch_ok,outputs/wyckoff_from_candidates_radius_mismatc...,16
91,Ni3Rh,mp-1189512,Eu->Ni,d,8,0.146580,accepted_radiusmatch_ok,outputs/wyckoff_from_candidates_radius_mismatc...,16
92,Mn3Rh,mp-1189512,Eu->Mn,d,8,0.125733,accepted_radiusmatch_ok,outputs/wyckoff_from_candidates_radius_mismatc...,16


In [9]:
import os
import numpy as np
import pandas as pd
from pymatgen.core import Structure


In [10]:
# Distance tolerance for shell grouping (same as paper)
DISTANCE_TOLERANCE = 0.1

# Max shells to consider
MAX_SHELLS = 4


In [11]:
def group_into_shells(neighbor_data, tol=DISTANCE_TOLERANCE):
    """
    Groups neighbors into distance-based coordination shells.
    """
    neighbor_data.sort(key=lambda x: x["distance"])

    shells = []
    current_shell = []

    for n in neighbor_data:
        if not current_shell:
            current_shell.append(n)
        else:
            if abs(n["distance"] - current_shell[0]["distance"]) <= tol:
                current_shell.append(n)
            else:
                shells.append(current_shell)
                current_shell = [n]

    if current_shell:
        shells.append(current_shell)

    return shells


In [12]:
def extract_environment(structure, central_index):
    """
    Extracts up to 4 coordination shells for a given central atom index.
    """
    central_site = structure[central_index]
    central_coords = np.array(central_site.coords)

    neighbor_data = []

    for i, site in enumerate(structure):
        if i == central_index:
            continue

        vec = np.array(site.coords) - central_coords
        dist = np.linalg.norm(vec)

        neighbor_data.append({
            "distance": dist,
            "species": site.species_string
        })

    shells = group_into_shells(neighbor_data)

    shell_info = []

    for shell in shells[:MAX_SHELLS]:
        species_set = set(n["species"] for n in shell)
        shell_info.append({
            "count": len(shell),
            "n_species": len(species_set),
            "species": ",".join(sorted(species_set))
        })

    return shell_info


In [13]:
def build_neighbor_pattern(shell_info):
    """
    Converts shell info into neighbor-count pattern string.
    """
    counts = [str(s["n_species"]) for s in shell_info]

    # Pad missing shells with NA
    while len(counts) < MAX_SHELLS:
        counts.append("NA")

    return "|".join(counts)


In [14]:
def process_cif(cif_path, metadata_row):
    """
    Processes one CIF file and returns species-centered environments.
    """
    structure = Structure.from_file(cif_path)

    results = []

    # Map species → representative site index
    species_to_index = {}
    for i, site in enumerate(structure):
        sp = site.species_string
        if sp not in species_to_index:
            species_to_index[sp] = i

    for species, idx in species_to_index.items():
        shell_info = extract_environment(structure, idx)
        pattern = build_neighbor_pattern(shell_info)

        row = {
            "candidate_formula": metadata_row["candidate_formula"],
            "prototype_id": metadata_row["prototype_id"],
            "substitution": metadata_row["substitution"],
            "wyckoff_letter": metadata_row["w_letter"],
            "central_species": species,
            "neighbor_pattern": pattern
        }

        # Optional: store raw shell data for debugging
        for i, shell in enumerate(shell_info, 1):
            row[f"shell{i}_count"] = shell["count"]
            row[f"shell{i}_species"] = shell["species"]

        results.append(row)

    return results


In [19]:
# def main():
#     candidates = pd.read_csv("accepted_reefree_structures.csv")

#     cif_dir = "wyckoff_from_candidates_radius_mismatch_recheck"
#     all_rows = []

#     for _, row in candidates.iterrows():
#         raw_path = row["generated_cif"]

#         # Fix path prefixes if needed
#         if raw_path.startswith("outputs/"):
#             raw_path = raw_path.replace("outputs/", "", 1)

#         cif_path = os.path.normpath(raw_path)

#         if not os.path.exists(cif_path):
#             print(f"Missing CIF: {cif_path}")
#             continue

#         try:
#             rows = process_cif(cif_path, row)
#             all_rows.extend(rows)
#         except Exception as e:
#             print(f"Failed on {cif_path}: {e}")

#     df = pd.DataFrame(all_rows)
#     df.to_csv("candidate_neighbor_configurations.csv", index=False)
#     print("Saved candidate_neighbor_configurations.csv")


def main():
    candidates = pd.read_csv("accepted_reefree_structures.csv")
    all_rows = []

    for _, row in candidates.iterrows():
        raw_path = row["generated_cif"]

        # Strip 'outputs' prefix if present
        if raw_path.startswith("outputs"):
            raw_path = raw_path.split("outputs", 1)[-1].lstrip("/\\")

        # Convert Windows path → macOS/Linux path
        raw_path = raw_path.replace("\\", "/")

        cif_path = os.path.normpath(raw_path)

        if not os.path.exists(cif_path):
            print(f"Missing CIF: {cif_path}")
            continue

        try:
            rows = process_cif(cif_path, row)
            all_rows.extend(rows)
        except Exception as e:
            print(f"Failed on {cif_path}: {e}")

    df = pd.DataFrame(all_rows)
    df.to_csv("candidate_neighbor_configurations.csv", index=False)
    print("Saved candidate_neighbor_configurations.csv")


In [20]:
if __name__ == "__main__":
    main()


Saved candidate_neighbor_configurations.csv


/Users/aditya/mtp_env/lib/python3.11/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 12 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/Users/aditya/mtp_env/lib/python3.11/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 14 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/Users/aditya/mtp_env/lib/python3.11/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 4 fractional coordinates rounded to ideal values to avoid issues with finite precision.
  struct = parser.parse_structures(primitive=primitive)[0]
/Users/aditya/mtp_env/lib/python3.11/site-packages/pymatgen/core/structure.py:3109: UserWarning: Issues encountered while parsing CIF: 20 fractional coo

In [23]:
dfff = pd.read_csv("candidate_neighbor_configurations.csv")
dfff.shape

(272, 14)